# Grammar Autocorrector: Staged LoRA Fine-Tuning of CoEdIT-Large
**Model:** `grammarly/coedit-large` (T5-Large, 770M Parameters)  
**Technique:** Low-Rank Adaptation (LoRA / PEFT) with Staged Curriculum Training  
**Environment:** Google Colab (Free T4 GPU) or Kaggle (T4x2 GPU)  

---

### Curriculum Overview
1. **Stage 1 (General Grammar, Spelling & Synthetic Noise):**
   - Data: `stage1_colab.csv` (150,000 pairs: 300k synthetic noise pairs + C4_200M + CoEdIT GEC, subsampled for cloud session)
   - Parameters: 1 epoch, learning rate `3e-4`, effective batch size 32 (8 per device x 4 accumulation steps)
   - Objective: Teach broad spelling, typo recovery, confusables, and fundamental syntax corrections.
2. **Stage 2 (In-Domain Fluency & Gold Polish):**
   - Data: `stage2.csv` (235,041 pairs: W&I+LOCNESS train + FCE v2.1 train + CoEdIT GEC, each gold pair oversampled x3)
   - Parameters: 3 epochs, learning rate `1e-4`, effective batch size 32
   - Objective: Master high-quality student writing style and error patterns.

> **Zero Data Leakage:** All evaluation sets (`jfleg_test.json` and `wi_dev.csv`) are strictly held out and verified with zero contamination.

## 1. Setup & Dependencies
Install Hugging Face Transformers, PEFT, Accelerate, Datasets, and SentencePiece.

In [ ]:
# Setup dependencies in Google Colab / Kaggle
!pip install -q transformers peft datasets accelerate sentencepiece pandas evaluate sacrebleu

import os, sys, shutil, json
from pathlib import Path
import pandas as pd
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Storage Setup (Google Drive / Kaggle Working)
Mount Google Drive so that training checkpoints and the final `gec-model.zip` are permanently saved even if your Colab session times out.

In [ ]:
# Configure persistent storage paths
if os.path.exists("/content"):
    # Google Colab
    from google.colab import drive
    drive_mount = "/content/drive"
    if not os.path.exists(f"{drive_mount}/MyDrive"):
        drive.mount(drive_mount)
    base_dir = Path(f"{drive_mount}/MyDrive/gec_training")
    print(f"[COLAB] Persistent Drive Directory: {base_dir}")
elif os.path.exists("/kaggle"):
    # Kaggle
    base_dir = Path("/kaggle/working/gec_training")
    print(f"[KAGGLE] Persistent Working Directory: {base_dir}")
else:
    base_dir = Path("./cloud_gec_training")

ckpt_dir = base_dir / "checkpoints"
export_dir = base_dir / "export"
ckpt_dir.mkdir(parents=True, exist_ok=True)
export_dir.mkdir(parents=True, exist_ok=True)

# Data directory resolution
# If you cloned the repository or uploaded data/processed, set DATA_DIR here
DATA_DIR = Path("data/processed")
if not DATA_DIR.exists():
    # Check common Colab / Kaggle upload locations
    for cand in [Path("/content/Grammar_Autocorrector/data/processed"), Path("/content/data/processed"), Path("/kaggle/working/data/processed")]:
        if cand.exists():
            DATA_DIR = cand
            break

print(f"Data Directory: {DATA_DIR}")
if not (DATA_DIR / "stage1_colab.csv").exists() and not (DATA_DIR / "stage1.csv").exists():
    print("\n[NOTICE] Please upload your 'data/processed' CSV files to Colab/Kaggle or clone your repository:")
    print("         !git clone <YOUR_REPO_URL>")

## 3. Hardware & Precision Configuration
Verify GPU device. Use `bf16` if supported by the GPU (Ampere/Hopper/etc.); otherwise use `fp32` (**T5 models are mathematically unstable under fp16**).

In [ ]:
assert torch.cuda.is_available(), "CUDA GPU is required for fine-tuning. Under Runtime > Change runtime type, select T4 GPU."

gpu_name = torch.cuda.get_device_name(0)
is_bf16 = torch.cuda.is_bf16_supported()
torch_dtype = torch.bfloat16 if is_bf16 else torch.float32

print("=" * 70)
print(f"GPU Detected:    {gpu_name}")
print(f"VRAM Available:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"bfloat16 Support: {is_bf16}")
print(f"Training Dtype:  {torch_dtype} (T5 fp16 instability avoided)")
print("=" * 70)

## 4. Tokenizer, Model & LoRA (PEFT) Setup
- **Base Model:** `grammarly/coedit-large`
- **Input Format:** `Fix grammatical errors in this sentence: <text>`
- **LoRA Parameters:** `r=16`, `alpha=32`, `dropout=0.05` on `q`, `k`, `v`, `o`, `wi`, `wo`
- **Gradient Checkpointing:** Enabled (drastically reduces VRAM footprint on 16GB T4)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL_NAME = "grammarly/coedit-large"
INPUT_PREFIX = "Fix grammatical errors in this sentence: "
MAX_LENGTH = 64

print(f"Loading tokenizer for {BASE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

print(f"Loading base model weights for {BASE_MODEL_NAME} ({torch_dtype})...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch_dtype,
)

# Configure LoRA adapter on attention + feed-forward projections
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q", "k", "v", "o", "wi", "wo"],
    bias="none",
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()
model.gradient_checkpointing_enable()

## 5. Dataset Definition & Validation Set Preparation
Validation is conducted each epoch on the combined benchmark of **W&I+LOCNESS dev** (`wi_dev.csv`) and **JFLEG dev** (`jfleg_dev.json`).

In [ ]:
from torch.utils.data import Dataset

class GECDataset(Dataset):
    def __init__(self, inputs, targets, tokenizer, max_length=MAX_LENGTH):
        self.inputs = [f"{INPUT_PREFIX}{str(inp).strip()}" for inp in inputs]
        self.targets = [str(tgt).strip() for tgt in targets]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        inp_enc = self.tokenizer(self.inputs[idx], max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        tgt_enc = self.tokenizer(self.targets[idx], max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        labels = tgt_enc.input_ids.squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": inp_enc.input_ids.squeeze(0),
            "attention_mask": inp_enc.attention_mask.squeeze(0),
            "labels": labels,
        }

# Load combined validation set
val_inputs, val_targets = [], []
wi_dev_path = DATA_DIR / "wi_dev.csv"
if wi_dev_path.exists():
    df_wi_dev = pd.read_csv(wi_dev_path, encoding="utf-8")
    val_inputs.extend(df_wi_dev["input"].astype(str).tolist())
    val_targets.extend(df_wi_dev["output"].astype(str).tolist())

jfleg_dev_path = DATA_DIR / "jfleg_dev.json"
if jfleg_dev_path.exists():
    with open(jfleg_dev_path, "r", encoding="utf-8") as f:
        jfleg_records = json.load(f)
    for rec in jfleg_records:
        if rec.get("source") and rec.get("references"):
            val_inputs.append(rec["source"])
            val_targets.append(rec["references"][0])

print(f"Total combined validation pairs: {len(val_inputs):,}")
val_dataset = GECDataset(val_inputs, val_targets, tokenizer)

## 6. Stage 1: Pre-Training on Broad Grammar, Spelling & Synthetic Noise
- **Dataset:** `stage1_colab.csv` (150,000 pairs subsampled for Colab sessions)
- **Hyperparameters:** 1 Epoch, Learning Rate `3e-4`, Batch Size 8 x 4 = 32
- **Auto-Resume & Checkpointing:** Checkpoint saved to Google Drive every 500 steps; resumes automatically if session reconnects.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
from transformers.trainer_utils import get_last_checkpoint

stage1_file = DATA_DIR / "stage1_colab.csv"
if not stage1_file.exists():
    stage1_file = DATA_DIR / "stage1.csv"

print(f"Loading Stage 1 data from {stage1_file}...")
df_stage1 = pd.read_csv(stage1_file, encoding="utf-8")
if len(df_stage1) > 150_000:
    df_stage1 = df_stage1.iloc[:150_000]
print(f"Stage 1 training pairs: {len(df_stage1):,}")

train_dataset_s1 = GECDataset(df_stage1["input"].astype(str).tolist(), df_stage1["output"].astype(str).tolist(), tokenizer)
stage1_ckpt_dir = ckpt_dir / "stage1"

stage1_args = Seq2SeqTrainingArguments(
    output_dir=str(stage1_ckpt_dir),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,  # 8 x 4 = 32 effective batch size
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    num_train_epochs=1,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    evaluation_strategy="epoch",
    save_safetensors=True,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    bf16=is_bf16,
    fp16=False,
    gradient_checkpointing=True,
    report_to="none",
)

trainer_s1 = Seq2SeqTrainer(
    model=model,
    args=stage1_args,
    train_dataset=train_dataset_s1,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Check for existing checkpoint to auto-resume
s1_checkpoint = get_last_checkpoint(str(stage1_ckpt_dir))
if s1_checkpoint:
    print(f"[RESUME] Found existing Stage 1 checkpoint at '{s1_checkpoint}'. Resuming...")
else:
    print("[START] Starting Stage 1 training...")

trainer_s1.train(resume_from_checkpoint=s1_checkpoint)

## 7. Stage 2: In-Domain Gold Refinement & Optional Targeted Mixing
- **Dataset:** `stage2.csv` (235,041 pairs: W&I Train + FCE Train + CoEdIT GEC oversampled x3)
- **Phase M6 Enhancement:** Optionally mix in `targeted.csv` (~30,000 pairs across top 10 weak error categories), keeping real gold data >= 50%.
- **Hyperparameters:** 3 Epochs, Learning Rate `1e-4`, Batch Size 8 x 4 = 32
- **Auto-Resume & Checkpointing:** Checkpoint saved to Google Drive every 500 steps.

In [ ]:
stage2_file = DATA_DIR / "stage2.csv"
print(f"Loading Stage 2 data from {stage2_file}...")
df_stage2 = pd.read_csv(stage2_file, encoding="utf-8")
print(f"Stage 2 base gold pairs: {len(df_stage2):,}")

# Phase M6: Optionally mix targeted.csv into Stage 2
USE_TARGETED_DATA = True  # Set to True to mix ~30k targeted error pairs
if USE_TARGETED_DATA:
    targeted_file = DATA_DIR / "targeted.csv"
    if not targeted_file.exists():
        targeted_file = Path("data/processed/targeted.csv")
    if targeted_file.exists():
        df_targeted = pd.read_csv(targeted_file, encoding="utf-8")
        print(f"[STAGE 2 MIXING] Loaded {len(df_targeted):,} targeted pairs from {targeted_file.name}")
        n_real = len(df_stage2)
        n_targeted = len(df_targeted)
        real_pct = (n_real / (n_real + n_targeted)) * 100
        print(f"[STAGE 2 MIXING] Real data: {n_real:,} | Targeted: {n_targeted:,} | Real %: {real_pct:.1f}% (Policy requirement: >= 50%)")
        assert real_pct >= 50.0, f"Real data ratio {real_pct:.1f}% is below required 50%!"
        df_stage2 = pd.concat([df_stage2[["input", "output"]], df_targeted[["input", "output"]]], ignore_index=True)
        df_stage2 = df_stage2.sample(frac=1.0, random_state=42).reset_index(drop=True)
        print(f"[STAGE 2 MIXING] Combined Stage 2 active pairs: {len(df_stage2):,}")
    else:
        print(f"[WARNING] USE_TARGETED_DATA is True but '{targeted_file}' not found. Continuing with pure stage2.csv.")

train_dataset_s2 = GECDataset(df_stage2["input"].astype(str).tolist(), df_stage2["output"].astype(str).tolist(), tokenizer)
stage2_ckpt_dir = ckpt_dir / "stage2"

stage2_args = Seq2SeqTrainingArguments(
    output_dir=str(stage2_ckpt_dir),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,  # 8 x 4 = 32 effective batch size
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=300,
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    evaluation_strategy="epoch",
    save_safetensors=True,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    bf16=is_bf16,
    fp16=False,
    gradient_checkpointing=True,
    report_to="none",
)

trainer_s2 = Seq2SeqTrainer(
    model=model,
    args=stage2_args,
    train_dataset=train_dataset_s2,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Check for existing checkpoint to auto-resume
s2_checkpoint = get_last_checkpoint(str(stage2_ckpt_dir))
if s2_checkpoint:
    print(f"[RESUME] Found existing Stage 2 checkpoint at '{s2_checkpoint}'. Resuming...")
else:
    print("[START] Starting Stage 2 training...")

trainer_s2.train(resume_from_checkpoint=s2_checkpoint)

## 8. Merge LoRA Weights & Export Standalone Model
Merge the trained LoRA adapter weights directly into the base T5 model weights (`merge_and_unload()`).  
This creates a standalone model in `gec-model/`, zips it to `gec-model.zip`, and displays the file size.

In [ ]:
print("=" * 70)
print("MERGING LoRA ADAPTER WEIGHTS INTO BASE MODEL")
print("=" * 70)

merged_model = model.merge_and_unload()
export_model_dir = export_dir / "gec-model"

print(f"Saving merged model to {export_model_dir}...")
merged_model.save_pretrained(str(export_model_dir))
tokenizer.save_pretrained(str(export_model_dir))

zip_base = base_dir / "gec-model"
print(f"Compressing merged model into '{zip_base}.zip'...")
shutil.make_archive(str(zip_base), "zip", root_dir=str(export_model_dir))

final_zip = base_dir / "gec-model.zip"
if final_zip.exists():
    size_mb = final_zip.stat().st_size / (1024 * 1024)
    print("\n" + "=" * 70)
    print(f"[SUCCESS] Distribution archive created: {final_zip}")
    print(f"Archive Size: {size_mb:.2f} MB ({size_mb / 1024:.2f} GB)")
    print("=" * 70)
    
    # In Colab, trigger direct browser download if possible:
    try:
        from google.colab import files
        files.download(str(final_zip))
    except Exception:
        print("Download 'gec-model.zip' from Google Drive or the Colab left file browser.")

## 9. Interactive Inference Verification
Run a few test sentences through the newly merged model to see the corrections live.

In [ ]:
merged_model.eval()
test_sentences = [
    "She go to school yesterday and buy a apple.",
    "Their going to there house because its late.",
    "I has three sister and we lives in Bengaluru.",
    "He is an honest man and works at university.",
    "Nowadays, students relies heavily onto internet for studies.",
]

print("=" * 70)
print("SAMPLE PREDICTIONS FROM MERGED GEC MODEL")
print("=" * 70)
for sent in test_sentences:
    prompt = f"{INPUT_PREFIX}{sent}"
    enc = tokenizer(prompt, return_tensors="pt", max_length=MAX_LENGTH, truncation=True).to(model.device)
    with torch.no_grad():
        gen = merged_model.generate(**enc, max_length=MAX_LENGTH, num_beams=2)
    pred = tokenizer.decode(gen[0], skip_special_tokens=True)
    print(f"Input:     {sent}")
    print(f"Corrected: {pred}\n")